In [4]:
import numpy as np
import pandas as pd
df_cycle = pd.read_csv('data/14_hydraulic.csv')
df_qc = pd.read_csv('data/14_hydraulic_qc.csv')

In [5]:
# coor: 두 값의 상관관계를 +1 ~ -1사이의 수치로 나타냄. 
# 부호의 음양은 상관관계가 반비례인지 비례인지 나타내고 
# 수치의 절대값은 상관관계의 크기를 나타낸다
# ex) df['온도'].corr(df['진동'])


In [ ]:
print(round(df_cycle['온도'].corr(df_cycle['진동']), 3))
print(round(df_cycle['온도'].corr(df_cycle['압력']),3))
print(round(df_cycle['온도'].corr(df_cycle['냉각효율']),3))
print('==='*15)
print(round(df_cycle[['온도','압력','냉각효율','진동']].corr(),3))
# 가장 강한 짝: 온도-냉각효율 = - 0.951, 가장 약한 짝: 압력-냉각효율 = -0.131

0.931
0.284
-0.951
         온도     압력   냉각효율     진동
온도    1.000  0.284 -0.951  0.931
압력    0.284  1.000 -0.131  0.524
냉각효율 -0.951 -0.131  1.000 -0.862
진동    0.931  0.524 -0.862  1.000


In [12]:
print(round(df_cycle.groupby('냉각기상태')[['온도','진동']].corr(),3))

             온도     진동
냉각기상태                 
고장    온도  1.000  0.701
      진동  0.701  1.000
저하    온도  1.000 -0.390
      진동 -0.390  1.000
정상    온도  1.000 -0.553
      진동 -0.553  1.000


In [ ]:
print(df_qc.columns)
correlation = round(df_qc[['온도', '진동', '압력', '냉각효율', '유량', '모터전력', '효율지수', '냉각출력', '배유온도']].corr(),3)
print(correlation)
# 짝은 36개
# 절댓값 0.4이상인 짝 = 33개
# 0.95 이상인 짝 = 11개
# 상위 5개 짝 = (유량,효율지수 = 0.999), (온도,배유온도 = 0.999), (온도,냉각효율 = -0.983), (유량,냉각출력 = 0.982), (냉각효율,배유온도 =-0.979)


Index(['검사결과', '온도', '진동', '압력', '냉각효율', '유량', '모터전력', '효율지수', '보조압력', '냉각출력',
       '배유온도'],
      dtype='object')
         온도     진동     압력   냉각효율     유량   모터전력   효율지수   냉각출력   배유온도
온도    1.000  0.941  0.716 -0.983 -0.895  0.327 -0.882 -0.960  0.999
진동    0.941  1.000  0.801 -0.940 -0.930  0.476 -0.920 -0.961  0.935
압력    0.716  0.801  1.000 -0.705 -0.950  0.885 -0.958 -0.878  0.696
냉각효율 -0.983 -0.940 -0.705  1.000  0.887 -0.333  0.873  0.949 -0.979
유량   -0.895 -0.930 -0.950  0.887  1.000 -0.709  0.999  0.982 -0.882
모터전력  0.327  0.476  0.885 -0.333 -0.709  1.000 -0.729 -0.570  0.300
효율지수 -0.882 -0.920 -0.958  0.873  0.999 -0.729  1.000  0.976 -0.868
냉각출력 -0.960 -0.961 -0.878  0.949  0.982 -0.570  0.976  1.000 -0.951
배유온도  0.999  0.935  0.696 -0.979 -0.882  0.300 -0.868 -0.951  1.000
9


In [ ]:

cols = ['온도', '진동', '압력', '냉각효율', '유량', '모터전력', '효율지수', '냉각출력', '배유온도']

correlation = round(df_qc[cols].corr(), 3)

mask = np.triu(np.ones(correlation.shape), k=1).astype(bool)

pairs = correlation.where(mask).stack()

total_pairs = len(pairs)

count_04 = (pairs.abs() >= 0.4).sum()

count_095 = (pairs.abs() >= 0.95).sum()

top5 = pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(5)

print("대각선을 뺀 전체 짝 개수:", total_pairs)
print("절댓값 0.4 이상인 짝 개수:", count_04)
print("절댓값 0.95 이상인 짝 개수:", count_095)

print("\n상위 다섯 짝:")
print(top5)

대각선을 뺀 전체 짝 개수: 36
절댓값 0.4 이상인 짝 개수: 33
절댓값 0.95 이상인 짝 개수: 11

상위 다섯 짝:
유량    효율지수    0.999
온도    배유온도    0.999
      냉각효율   -0.983
유량    냉각출력    0.982
냉각효율  배유온도   -0.979
dtype: float64


In [28]:
df_cycle = pd.read_csv('data/14_hydraulic.csv')
df_std = pd.read_csv('data/14_hydraulic_std.csv')
# 0~40, 41~50, 51~200
print(df_cycle.shape, df_std.shape)

df_merged = df_cycle.merge(df_std, on='냉각기상태', how='left')
print(df_merged.shape)
df_merged_copy = df_merged.copy()
temps_band = pd.cut(df_merged['온도'], bins=[0, 40, 50, 200], labels=['low', 'mid', 'high'])
df_merged_copy['온도구간'] = pd.cut(df_merged_copy['온도'], bins=[0, 40, 50, 200], labels=['low', 'mid', 'high'])
print(temps_band.value_counts())

summary = df_merged.groupby('냉각기상태').agg(
    건수=('온도', 'count'),
    평균온도=('온도', 'mean'),
    온도편차=('온도', 'std'),
    평균냉각효율=('냉각효율', 'mean'),
    위험도=('위험도', 'max'),
    점검주기=('권장점검주기', 'max')
).round(2).reset_index()

print(summary)

(120, 8) (4, 5)
(120, 12)
온도
low     41
mid     40
high    39
Name: count, dtype: int64
  냉각기상태  건수   평균온도  온도편차  평균냉각효율  위험도  점검주기
0    고장  40  54.67  3.76   20.12    3     7
1    저하  40  45.46  1.47   27.24    2    60
2    정상  40  35.89  0.36   46.96    1   180


In [ ]:
df_merged = df_shot.merge(df_master, on='설비ID', how='left')

In [32]:
print(pd.crosstab(df_merged_copy['냉각기상태'], df_merged_copy['온도구간']))
print(round(df_merged_copy.corr[['온도','냉각효율','진동']],3))

온도구간   low  mid  high
냉각기상태                
고장       1    1    38
저하       0   39     1
정상      40    0     0


TypeError: 'method' object is not subscriptable